# Step 3: SQL Funnel & Cohort View Engine

**Objective:** Create production SQL views inside MySQL to transform raw clickstream events into structured analytical datasets for funnel drop-off analysis and cohort retention.

**Key Analytical Views Built:**
1. `v_funnel_conversion_rates`: Overall stage-by-stage funnel performance.
2. `v_channel_performance`: Funnel drop-offs segmented by acquisition channel.
3. `v_monthly_cohort_retention`: Monthly cohort retention metrics.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from urllib.parse import quote_plus

# Load environment variables
load_dotenv(override=True)

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1').strip()
DB_PORT = os.getenv('DB_PORT', '3306').strip()
DB_NAME = os.getenv('DB_NAME')

encoded_password = quote_plus(DB_PASSWORD)
connection_string = f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string, pool_pre_ping=True)

print("✅ Successfully reconnected to MySQL database!")

✅ Successfully reconnected to MySQL database!


In [2]:
create_view_2_sql = """
CREATE OR REPLACE VIEW v_channel_performance AS
WITH user_channel_flags AS (
    SELECT 
        user_id,
        acquisition_channel,
        MAX(CASE WHEN event_type = 'signup' THEN 1 ELSE 0 END) AS signed_up,
        MAX(CASE WHEN event_type = 'activated' THEN 1 ELSE 0 END) AS activated,
        MAX(CASE WHEN event_type = 'purchased' THEN 1 ELSE 0 END) AS purchased
    FROM events
    GROUP BY user_id, acquisition_channel
)
SELECT 
    acquisition_channel,
    COUNT(*) AS total_signups,
    SUM(activated) AS total_activations,
    SUM(purchased) AS total_purchases,
    ROUND(SUM(activated) / COUNT(*) * 100, 2) AS activation_rate_pct,
    ROUND(SUM(purchased) / SUM(activated) * 100, 2) AS purchase_rate_pct,
    ROUND(SUM(purchased) / COUNT(*) * 100, 2) AS overall_conversion_pct
FROM user_channel_flags
GROUP BY acquisition_channel
ORDER BY overall_conversion_pct DESC;
"""

with engine.connect() as conn:
    conn.execute(text(create_view_2_sql))
    conn.commit()

# Verify View Output
channel_summary = pd.read_sql("SELECT * FROM v_channel_performance;", con=engine)
print("--- Channel Performance Summary ---")
channel_summary

--- Channel Performance Summary ---


,acquisition_channel,total_signups,total_activations,total_purchases,activation_rate_pct,purchase_rate_pct,overall_conversion_pct
0,Email,9458,5700.0,956.0,60.27,16.77,10.11
1,Organic Search,33804,20157.0,3187.0,59.63,15.81,9.43
2,Direct,9661,5858.0,900.0,60.64,15.36,9.32
3,Social Media,19339,11616.0,1790.0,60.07,15.41,9.26
4,Paid Search,23834,14224.0,2196.0,59.68,15.44,9.21


In [3]:
create_view_3_sql = """
CREATE OR REPLACE VIEW v_monthly_cohort_retention AS
WITH user_cohorts AS (
    SELECT 
        user_id,
        DATE_FORMAT(MIN(event_timestamp), '%Y-%m-01') AS cohort_month
    FROM events
    WHERE event_type = 'signup'
    GROUP BY user_id
),
user_activities AS (
    SELECT 
        e.user_id,
        uc.cohort_month,
        DATE_FORMAT(e.event_timestamp, '%Y-%m-01') AS activity_month,
        PERIOD_DIFF(
            DATE_FORMAT(e.event_timestamp, '%Y%m'), 
            DATE_FORMAT(uc.cohort_month, '%Y%m')
        ) AS month_number
    FROM events e
    JOIN user_cohorts uc ON e.user_id = uc.user_id
)
SELECT 
    cohort_month,
    month_number,
    COUNT(DISTINCT user_id) AS active_users
FROM user_activities
GROUP BY cohort_month, month_number
ORDER BY cohort_month, month_number;
"""

with engine.connect() as conn:
    conn.execute(text(create_view_3_sql))
    conn.commit()

# Verify View Output
cohort_summary = pd.read_sql("SELECT * FROM v_monthly_cohort_retention LIMIT 10;", con=engine)
print("--- Monthly Cohort Retention Sample ---")
cohort_summary

--- Monthly Cohort Retention Sample ---


,cohort_month,month_number,active_users
0,2017-01-01,0,5473
1,2017-01-01,1,13
2,2017-02-01,0,4900
3,2017-02-01,1,26
4,2017-03-01,0,5548
5,2017-03-01,1,27
6,2017-04-01,0,5296
7,2017-04-01,1,19
8,2017-05-01,0,5433
9,2017-05-01,1,25
